<a href="https://colab.research.google.com/github/paymantohidifar/build-a-llm-from-scratch-book/blob/main/llms-from-scratch/notebooks/ch03/01_main-chapter-code/ch03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

## **Local Setup**

Run the two cells below to verify that the correct environment loaded. Ignore this if you're on Google Colab session.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

executable_path = Path(sys.executable)
active_env_idx = executable_path.parts.index("envs")
assert executable_path.parts[active_env_idx+1] == "default", "Wrong environment. Choose 'default' environment."
print("'Default' environment is loaded successfully!")

## **Google Colab Setup**

Please uncomment the cell below and run it to install all dependencies and resolve import paths. After installation is completed, restart your session to clear Colab's previous Python cache.

**CPU vs. GPU**: To install CUDA-enabled PyTorch, start your session with GPU runtime session and replace `uv pip install -e .[cpu,dev] ...` with `uv pip install -e .[gpu,dev] ...` in the fillowing cell.  

In [ ]:
# # Clone the repo locally
# !rm -rf /content/build-a-llm-from-scratch-book
# !git clone https://github.com/paymantohidifar/build-a-llm-from-scratch-book.git --branch main
# %cd build-a-llm-from-scratch-book/llms-from-scratch

# # Bootstrap uv globally and pull GPU-enabled binaries directly into the system layer
# !curl -LsSf https://astral.sh/uv/install.sh | sh && \
# export PATH="$HOME/.local/bin:${PATH}" && \
# uv pip install -e .[gpu,dev] \
#         --system \
#         --break-system-packages \
#         --color never

# # Add `src/` to system path for local imports
# import sys
# sys.path.append('/content/build-a-llm-from-scratch-book/llms-from-scratch/src') # Point to the cloned directory

## **Verify Installation of Necessary Packages**

In [ ]:
from importlib.metadata import version

print(f"PyTorch version: {version("torch")}")

---

# **Chapter 3: Coding Attention Mechanisms**

---

## **Table of Contents**

- [Introduction](#Introduction)
- [3.1 The problem with modeling long sequences](#3-1-the-problem-with-modeling-long-sequences)
- [3.2 Capturing data dependencies with attention mechanisms](#3-2-capturing-data-dependencies-with-attention-mechanisms)
- [3.3 Attending to different parts of the input with self-attention](#3-3-attending-to-different-parts-of-the-input-with-self-attention)
    - [3.3.1 A simple self-attention mechanism without trainable weights](#3-3-1-a-simple-self-attention-mechanism-without-trainable-weights)
    - [3.3.2 Computing attention weights for all input tokens](#3-3-2-computing-attention-weights-for-all-input-tokens)
- [3.4 Implementing self-attention with trainable weights](#3-4-implementing-self-attention-with-trainable-weights)
    - [3.4.1 Computing the attention weights step by step](#3-4-1-computing-the-attention-weights-step-by-step)
    - [3.4.2 Implementing a compact SelfAttention class](#3-4-2-implementing-a-compact-selfattention-class)
- [3.5 Hiding future words with causal attention](#3-5-hiding-future-words-with-causal-attention)
    - [3.5.1 Applying a causal attention mask](#3-5-1-applying-a-causal-attention-mask)
    - [3.5.2 Masking additional attention weights with dropout](#3-5-2-masking-additional-attention-weights-with-dropout)
    - [3.5.3 Implementing a compact causal self-attention class](#3-5-3-implementing-a-compact-causal-self-attention-class)
- [3.6 Extending single-head attention to multi-head attention](#3-6-extending-single-head-attention-to-multi-head-attention)
    - [3.6.1 Stacking multiple single-head attention layers](#3-6-1-stacking-multiple-single-head-attention-layers)
    - [3.6.2 Implementing multi-head attention with weight splits](#3-6-2-implementing-multi-head-attention-with-weight-splits)
- [Summary and takeaways](#Summary-and-takeaways)
- [Supoplementary Information](#Supoplementary-Information)

---

## **Introduction**

In this chapter, we will transition from data preparation to implementing **attention mechanisms**, which are described as an "integral part" and the "cornerstone" of the transformer-based LLM architecture. The primary objective of the chapter is to develop a functional **multi-head attention module** that can be integrated into the full GPT architecture in later chapters.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.1</strong>  The three main stages of coding an LLM. This chapter focuses on step 2 of stage 1: implementing attention mechanisms, which are an integral part of the LLM architecture.
    </figcaption>
</figure>

We will progress through four distinct variants of attention that build upon one another (**Fig. 3.2**):

*   **Simplified self-attention**: This initial version is free from trainable weights and is used to introduce the fundamental concept of computing context vectors by relating different positions within a single input sequence.
*   **Self-attention**: This variant adds **trainable weight matrices**, allowing the model to learn and optimize how it relates tokens. This forms the standard "scaled dot-product attention" used in modern LLMs.
*   **Causal attention**: Also known as masked attention, this version restricts the model to only consider previous and current tokens in a sequence. By masking future tokens, it ensures the model maintains correct temporal order during text generation.
*   **Multi-head attention**: The most advanced variant, it organizes the mechanism into multiple parallel "heads". This allows the model to simultaneously attend to information from different representation subspaces, capturing various aspects of the input data at once.

Ultimately, this step-by-step implementation is designed to conquer one of the most complex aspects of building an LLM from scratch.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/02.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.2</strong>  The figure depicts different attention mechanisms we will code in this chapter, starting with a simplified version of self-attention before adding the trainable weights. The causal attention mechanism adds a mask to self-attention that allows the LLM to generate one word at a time. Finally, multi-head attention organizes the attention mechanism into multiple heads, allowing the model to capture various aspects of the input data in parallel.
    </figcaption>
</figure>

## **3.1 The problem with modeling long sequences**

This section explains the technical limitations of pre-transformer architectures, specifically Recurrent Neural Networks (RNNs), in processing language.

The core issues detailed in this section include:

* **Translation Challenges**: Using word-by-word translation is ineffective because grammatical structures vary between languages; therefore, a model must be able to "look back" at various parts of the input to ensure accuracy (**Fig. 3.3**).

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.3</strong> When translating text from one language to another, such as German to English, it’s not possible to merely translate word by word. Instead, the translation process requires contextual understanding and grammatical alignment.
    </figcaption>
</figure>

* **Sequential Bottlenecks**: In traditional encoder-decoder RNNs, the encoder must compress the entire meaning of an input sequence into a single hidden state (memory cell) before passing it to the decoder (**Fig. 3.4**).

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.4</strong> Before the advent of transformer models, encoder–decoder RNNs were a popular choice for machine translation. The encoder takes a sequence of tokens from the source language as input, where a hidden state (an intermediate neural network layer) of the encoder encodes a compressed representation of the entire input sequence. Then, the decoder uses its current hidden state to begin the translation, token by token.
    </figcaption>
</figure>

* **Loss of Context**: Because the decoder only has access to this final hidden state rather than the original tokens, the model often loses vital context. This is especially problematic for complex or long sentences where grammatical dependencies span great distances.

Ultimately, this inability to directly access earlier information during the decoding phase served as the primary motivation for the development of attention mechanisms, which solve this problem by allowing models to focus on specific parts of the input sequence regardless of their position.

---

## **3.2 Capturing data dependencies with attention mechanisms**

This section discusses the evolution from traditional recurrent architectures to the attention-based models used today.

The key developments highlighted in this section include:

* **RNN Limitations**: While Recurrent Neural Networks (RNNs) work for short sequences, they fail on longer texts because they must compress the entire encoded input into a single hidden state. This lack of direct access to earlier tokens often leads to a loss of vital context.
* **The Bahdanau Mechanism**: Developed in 2014, this was the first major step toward solving the bottleneck. It modified RNNs to allow the decoder to selectively access different parts of the input sequence at each decoding step (**Fig. 3.5**).

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/05.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.5</strong>  Using an attention mechanism, the text-generating decoder part of the network can access all input tokens selectively. This means that some input tokens are more important than others for generating a given output token. The importance is determined by the attention weights, which we will compute later. Note that this figure shows the general idea behind attention and does not depict the exact implementation of the Bahdanau mechanism, which is an RNN method outside this book’s scope.
    </figcaption>
</figure>

* **The Shift to Self-Attention**: By 2017, researchers realized that RNN architectures were not necessary. They proposed the transformer architecture, which utilizes a self-attention mechanism inspired by the Bahdanau approach.
* **Defining Self-Attention**: This mechanism allows each position in an input sequence to weigh the importance of and interact with all other positions within that same sequence. This capability is the cornerstone of modern LLMs like the GPT series.

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/06.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.6</strong> Self-attention is a mechanism in transformers used to compute more efficient input representations by allowing each position in a sequence to interact with and weigh the importance of all other positions within the same sequence. In this chapter, we will code this self-attention mechanism from the ground up before we code the remaining parts of the GPT-like LLM in the following chapter.
    </figcaption>
</figure>

Ultimately, this section establishes that the transition to self-attention was driven by the need to capture long-range dependencies more effectively and efficiently than sequential RNNs could.

---

## **3.3 Attending to different parts of the input with self-attention**

This section introduces the "cornerstone" mechanism of transformer-based LLMs that allows each position in a sequence to interact with and weigh the importance of all other positions within that same sequence. Unlike traditional attention mechanisms that relate two different sequences, the "self" in self-attention refers to analyzing relationships within a single input sequence.

The section illustrates the process through a simplified version without trainable weights, focusing on three core computational steps to produce context vectors, which are enriched representations of input tokens:

* **Compute Attention Scores ($\omega$):** The model uses the dot product to measure the similarity or alignment between a chosen "query" token and every other token in the sequence. A higher dot product signifies that the tokens are more closely related in context.
* **Normalize Scores into Weights ($\alpha$):** These scores are normalized, typically using the softmax function, so that they are always positive and sum to 1. This allows the weights to be interpreted as the relative importance or probability of each token in the context of the query.
* **Calculate Context Vectors ($z$):** The final context vector is a weighted sum of all input vectors, where each input is multiplied by its corresponding normalized attention weight. This creates an enriched embedding for each token that incorporates relevant information from across the entire sequence.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.7</strong> The goal of self-attention is to compute a context vector for each input element that combines information from all other input elements. In this example, the importance or contribution of each input element for computing context vector for second element is determined by the attention weights. When computing context vector for the second element, the attention weights are calculated with respect to the second element and all other input elements. Please note that the numbers in this figure are truncated to one digit after the decimal point to reduce visual clutter; similarly, other figures may also contain truncated values.
    </figcaption>
</figure>


While these steps are initially shown using sequential loops for clarity, the section emphasizes that they are performed simultaneously and much more efficiently in actual models using matrix multiplications. This mathematical optimization allows the model to compute context vectors for all tokens in an entire batch in parallel.

### **3.3.1 A simple self-attention mechanism without trainable weights**

This section explains a very simplified variant of self-attention, which does not contain any trainable weights. This is purely for illustration purposes and NOT the attention mechanism that is used in transformers. The next section, section 3.3.2, will extend this simple attention mechanism to implement the real self-attention mechanism.

Suppose we are given an input sequence $x^{(1)}$ to $x^{(T)}$. The input is a text (for example, a sentence like "Your journey starts with one step") that has already been converted into token embeddings as described in chapter 2. For instance, $x^{(1)}$ is a d-dimensional vector representing the word "Your", and so forth. The goal is to compute context vectors $z^{(i)}$ for each input sequence element $x^{(i)}$ in $x^{(1)}$ to $x^{(T)}$ (where $z$ and $x$ have the same dimension).


**Step 1:** compute unnormalized attention scores $\omega$. Suppose we use the second input token as the query, that is, $q^{(2)} = x^{(2)}$, we compute the unnormalized attention scores via dot products:
* $\omega_{21} = x^{(1)} q^{(2)\top}$
* $\omega_{22} = x^{(2)} q^{(2)\top}$
* $\omega_{23} = x^{(3)} q^{(2)\top}$
* $\dots$
* $\omega_{2T} = x^{(T)} q^{(2)\top}$
The subscript "21" in $\omega_{21}$ means that input sequence element 2 was used as a query against input sequence element 1

Suppose we have the following input sentence that is already embedded in 3-dimensional vectors as described in chapter 3 (we use a very small embedding dimension here for illustration purposes, so that it fits onto the page without line breaks).

>**Note:** In this book, we follow the common machine learning and deep learning convention where training examples are represented as rows and feature values as columns; in the case of the tensor shown above, each row represents a word, and each column represents an embedding dimension.

In [ ]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

The primary objective of this section is to demonstrate how the context vector $z^{(2)}$ is calculated using the second input sequence, $x^{(2)}$, as a query. The figure depicts the initial step in this process, which involves calculating the attention scores ω between $x^{(2)}$ and all other input elements through a dot product operation.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.8</strong> The overall goal is to illustrate the computation of the context vector for the second element using the second input element as a query. This figure shows the first intermediate step, computing the attention scores between the query and all other input elements as a dot product. (Note that the numbers are truncated to one digit after the decimal point to reduce visual clutter.)
    </figcaption>
</figure>

In [ ]:
query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

**Step 2:** normalize the unnormalized attention scores ("omegas", $\omega$) so that they sum up to 1. Here is a simple way to normalize the unnormalized attention scores to sum up to 1 (a convention, useful for interpretation, and important for training stability):

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.9</strong> After computing the attention scores with respect to the input query 2, the next step is to obtain the attention weights by normalizing the attention scores.
    </figcaption>
</figure>

In [ ]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

In practice, using the softmax function for normalization, which is better at handling extreme values and has more desirable gradient properties during training, is common and recommended. Here's a naive implementation of a softmax function for scaling, which also normalizes the vector elements such that they sum up to 1:

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

The naive implementation above can suffer from numerical instability issues for large or small input values due to overflow and underflow issues. Hence, in practice, it's recommended to use the PyTorch implementation of softmax instead, which has been highly optimized for performance:

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

**Step 3**: compute the context vector $z^{(2)}$ by multiplying the embedded input tokens, $x^{(i)}$ with the attention weights and sum the resulting vectors:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.10</strong> The final step, after calculating and normalizing the attention scores to obtain the attention weights for query 2, is to compute the context vector. This context vector is a combination of all input vectors weighted by the attention weights.
    </figcaption>
</figure>

In [ ]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

### **3.3.2 Computing attention weights for all input tokens**

Above, we computed the attention weights and context vector for input 2 (as illustrated in the highlighted row in the figure below). Next, we are generalizing this computation to compute all attention weights and context vectors:

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/11.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.11</strong> The highlighted row shows the attention weights for the second input element as a query.
    </figcaption>
</figure>

Please note that the numbers in this figure are truncated to two digits after the decimal point to reduce visual clutter; the values in each row should add up to 1.0 or 100%; similarly, digits in other figures are truncated.

Now we apply previous **step 1** to all pairwise elements to compute the unnormalized attention score matrix:

In [ ]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

We can achieve the same as above more efficiently via matrix multiplication:

In [ ]:
attn_scores = inputs @ inputs.T
# or
# attn_scores = torch.matmul(inputs, inputs.T)
print(attn_scores)

Similar to **step 2** previously, we normalize each row so that the values in each row sum to 1:

In [ ]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

Quick verification that the values in each row indeed sum to 1:

In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Apply previous **step 3** to compute all context vectors:

In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

As a sanity check, the previously computed context vector $z^{(2)} = [0.4419, 0.6515, 0.5683]$ can be found in the 2nd row in above:

In [ ]:
assert torch.allclose(context_vec_2, all_context_vecs[1], atol=1e-6)

---

## **3.4 Implementing self-attention with trainable weights**

This section introduces the **scaled dot-product attention** mechanism, which is the foundational component used in the original transformer, GPT models, and most modern LLMs.

The key distinction of this version compared to simplified self-attention is the introduction of trainable weight matrices that are updated during the model's training process. These matrices are crucial because they enable the attention module to learn and optimize how it transforms input data into effective context vectors. The source specifies that this implementation will be explored in two stages: first through a step-by-step computational walkthrough and then by organizing the logic into a compact Python class suitable for an LLM architecture.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/13.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.13</strong> A conceptual framework illustrating how the self-attention mechanism developed in this section integrates into the overall narrative and structure of this book and chapter.
    </figcaption>
</figure>


### **3.4.1 Computing the attention weights step by step**

This subsection details the mathematical workflow of the scaled dot-product attention mechanism, the standard version used in modern LLMs like GPT.

The core process introduces three trainable weight matrices—$W_q$ (Query), $W_k$ (Key), and $W_v$ (Value)—which allow the model to learn and optimize how it transforms input data into context vectors.

The computational steps are as follows:

* **Projection into Q, K, and V**: Each input embedding ($x$) is multiplied by the three weight matrices to produce distinct **query ($q$)**, **key ($k$)**, and **value ($v$)** vectors. QKV terms are borrowed from database management to describe the attention mechanism's internal operations:
  * **Query ($q$):** Represents the current token the model is focusing on to understand its relationship with the rest of the sequence.
    * Query vector: $q^{(i)} = x^{(i)}\,W_q $
  * **Key ($k$):** Functions like a database index used for matching; it is compared against the query to determine relevance.
    * Key vector: $k^{(i)} = x^{(i)}\,W_k $
  * **Value ($v$):** Represents the actual information or "content" of the input items that will be retrieved once relevance is established.
    * Value vector: $v^{(i)} = x^{(i)}\,W_v $

  *The embedding dimensions of the input $x$ and the query vector $q$ can be the same or different, depending on the model's design and specific implementation. In GPT models, the input and output dimensions are usually the same.*

  <figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.14</strong> Three training weight matrices are used to project the embedded input tokens into query, key, and value vectors via matrix multiplication.
    </figcaption>
  </figure>


* **Calculating Attention Scores**: The model computes unnormalized attention scores ($\omega$) by taking the dot product between a specific query vector and all available key vectors in the sequence.
* **Scaling for Stability**: To maintain training stability, the attention scores are divided by the square root of the key dimension ($\sqrt{d_k}$). This scaling prevents gradients from becoming vanishingly small as embedding dimensions increase, which would otherwise slow or stagnate learning (prevents softmax behaving as a step function for large attention scors).
* **Normalization**: The scaled scores are passed through a softmax function to produce normalized attention weights ($\alpha$) that sum to 1.
* **Context Vector Generation**: The final context vector ($z$) is computed as a weighted sum of all value vectors, using the normalized attention weights to determine the relative importance of each value.

Ultimately, this framework allows the LLM to create enriched representations by "attending" to the most relevant information across the input sequence based on learned parameters.

For illustration purpose, we will start with a small token with embedding size of 3. The goal is to generate a context vector with embedding size of 2:

In [ ]:
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2
x_2

Below, we initialize the three weight matrices; note that we are setting `requires_grad=False` to reduce clutter in the outputs for illustration purposes, but if we were to use the weight matrices for model training, we would set `requires_grad=True` to update these matrices during model training.

In [ ]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

Next we compute the query, key, and value vectors:

In [ ]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

As we can see below, we successfully projected the 6 input tokens from a 3D onto a 2D embedding space:

In [ ]:
keys = inputs @ W_key
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

In the next step, **step 2**, we compute the unnormalized attention scores by computing the dot product between the query and each key vector:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.15</strong> The attention score computation is a dot-product computation similar to what we used in the simplified self-attention mechanism in section 3.3. The new aspect here is that we are not directly computing the dot-product between the input elements but using the query and key obtained by transforming the inputs via the respective weight matrices.
    </figcaption>
</figure>

In [ ]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2) # 22: Query 2 attending key 2
print(attn_score_22)

Since we have 6 inputs, we have 6 attention scores for the given query vector:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.16</strong> After computing the attention scores, the next step is to normalize these scores using the softmax function to obtain the attention weights.
    </figcaption>
</figure>

In [ ]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

Next, in **step 3**, we compute the attention weights (normalized attention scores that sum up to 1) using the softmax function we used earlier. The difference to earlier is that we now scale the attention scores by dividing them by the square root of the embedding dimension, $\sqrt{d_k}$ (i.e., `d_k**0.5`):

In [ ]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

In **step 4**, we now compute the context vector for input query vector 2:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.17</strong>  In the final step of the self-attention computation, we compute the context vector by combining all value vectors via the attention weights.
    </figcaption>
</figure>

In [ ]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

### **3.4.2 Implementing a compact SelfAttention class**

This section focuses on organizing the step-by-step logic of scaled dot-product attention into reusable PyTorch modules suitable for a full LLM architecture. The section details two primary implementations and the conceptual framework supporting them:

1. `SelfAttention_v1`: Manual Parameter Management
This initial class implementation is derived from `nn.Module`, the fundamental building block for PyTorch layers. It manually initializes three trainable weight matrices ($W_q$, $W_k$, and $W_v$) using `nn.Parameter` and random tensors. The `forward` method then executes the computational pipeline: projecting inputs into Q, K, and V vectors, calculating scaled attention scores, applying the softmax function, and finally generating context vectors.

2. `SelfAttention_v2`: Optimized Linear Layers
The source introduces an improved version that utilizes `nn.Linear` layers instead of manual parameters. This is the preferred implementation for two key reasons:
    * **Weight Initialization**: Unlike manual random tensors, `nn.Linear` includes optimized initialization schemes that contribute to more stable and effective training dynamics.
    * **Computational Efficiency**: These layers are highly optimized within the PyTorch framework to handle the matrix multiplications necessary for processing entire sequences in parallel.

Ultimately, these classes allow the model to transform a matrix of input embeddings into a matrix of **enriched context vectors**, where each row captures the semantic relationships of a token relative to the entire sequence.

Putting it all together, we can implement the first version of self-attention mechanism as follows:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.18</strong>
    </figcaption>
</figure>

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

We can streamline the implementation above using PyTorch's Linear layers, which are equivalent to a matrix multiplication if we disable the bias units:

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
att = SelfAttention_v2(d_in, d_out)
att(inputs)

att.W_query.weight.T

In [ ]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

Note that `SelfAttention_v1` and `SelfAttention_v2` give different outputs because they use different initial weights for the weight matrices

#### **Exercise 3.1 Comparing `SelfAttention_v1` and `SelfAttention_v2`**

Note that `nn.Linear` in `SelfAttention_v2` uses a different weight initialization scheme as `nn.Parameter(torch.rand(d_in, d_out))` used in `SelfAttention_v1`, which causes both mechanisms to produce different results. To check that both implementations, `SelfAttention_v1` and `SelfAttention_v2`, are otherwise similar, we can transfer the weight matrices from a `SelfAttention_v2` object to a `SelfAttention_v1`, such that both objects then produce the same results.

Your task is to correctly assign the weights from an instance of `SelfAttention_v2` to an instance of `SelfAttention_v1` . To do this, you need to understand the relationship between the weights in both versions. (Hint: `nn.Linear` stores the weight matrix in a transposed form.) After the assignment, you should observe that both instances produce the same outputs.

#### **Solution 3.1:**

In [ ]:
sa_v1.W_key = nn.Parameter(sa_v2.W_key.weight.T, requires_grad=False)
sa_v1.W_query = nn.Parameter(sa_v2.W_query.weight.T, requires_grad=False)
sa_v1.W_value = nn.Parameter(sa_v2.W_value.weight.T, requires_grad=False)

v1_output = sa_v1(inputs)
v2_output = sa_v2(inputs)

assert torch.allclose(v1_output, v2_output), "V1 and V2 outputs are different"
print("Output of v1 and V2 are now equal.")

## **3.5 Hiding future words with causal attention**

This section introduces a specialized form of self-attention called **causal attention** (or **masked attention**), which is essential for language modeling such as training GPT-like Large Language Models to generate text.

The key technical and functional aspects include:

* **Restricted Context**: Unlike standard self-attention, which allows access to the entire sequence, causal attention restricts the model to only consider previous and current tokens when processing any given word. This ensures the model maintains a correct temporal order during text generation.
* **The Masking Mechanism**: In GPT-like architectures, this is implemented by masking out "future" tokens that appear after the current position in the input text.
* **Diagonal Masking**: Mathematically, the model zeroes out all attention weights located above the diagonal of the attention weight matrix.
* **Normalization**: After masking, the remaining non-zero attention weights in each row are renormalized so they sum to 1, ensuring mathematical consistency for the next stages of the transformer block.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.19</strong> Causal self attention using diagonal masking. For example, for the word “journey” in the second row, we only keep the attention
weights for the words before (“Your”) and in the current position (“journey”).
    </figcaption>
</figure>

Ultimately, this mechanism enables the LLM to learn unidirectional, left-to-right processing, allowing it to predict subsequent tokens based solely on the preceding context.

### **3.5.1 Applying a causal attention mask**

This section details the programmatic implementation of the masking mechanism required for text generation in GPT-like models.

Two primary methods for achieving causal masking is discussed:

* **The Manual Three-Step Approach**:
    1.  Compute standard attention weights using the softmax function.
    2.  Create a mask using PyTorch’s `tril` function** to zero out all values above the diagonal (representing future tokens).
    3.  Renormalize the remaining non-zero weights by dividing them by their respective row sums so that each row sums to 1 again.
* **The Efficient "Masking Trick"**: A more computationally elegant method involves masking the unnormalized attention scores with negative infinity (`-inf`) above the diagonal *before* applying the softmax function. Because $e^{-\infty}$ approaches zero, the softmax function automatically assigns these positions a probability of zero and ensures the remaining weights are perfectly normalized.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.20</strong> One way to obtain the masked attention weight matrix in causal attention is to apply the softmax function to the attention scores, zeroing out the elements above the diagonal and normalizing the resulting matrix.
    </figcaption>
</figure>

Ultimately, this implementation ensures there is no "information leakage" from future tokens, allowing the model to focus strictly on the current and preceding context during training and inference.

To illustrate and implement causal self-attention, let's work with the attention scores and weights from the previous section:

In [ ]:
# Reuse the query and key weight matrices of the
# SelfAttention_v2 object from the previous section for convenience
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

The simplest way to mask out future attention weights is by creating a mask via PyTorch's `tril` function with elements below the main diagonal (including the diagonal itself) set to 1 and above the main diagonal set to 0:

In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

Then, we can multiply the attention weights with this mask to zero out the attention scores above the diagonal:

In [ ]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

However, if the mask were applied after softmax, like above, it would disrupt the probability distribution created by softmax. Masking after softmax would require re-normalizing the outputs to sum to 1 again, which complicates the process and might lead to unintended effects:

To make sure that the rows sum to 1, we can normalize the attention weights as follows:

In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

While we are technically done with coding the causal attention mechanism now, let's briefly look at a more efficient approach to achieve the same as above. So, instead of zeroing out attention weights above the diagonal and renormalizing the results, we can mask the unnormalized attention scores above the diagonal with negative infinity before they enter the softmax function:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.21</strong> A more efficient way to obtain the masked attention weight matrix in causal attention is to mask the attention scores with negative infinity values before applying the softmax function.
    </figcaption>
</figure>

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

As we can see below, now the attention weights in each row correctly sum to 1 again:

In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

### **3.5.2 Masking additional attention weights with dropout**

This section explains how to integrate the dropout regularization technique into the attention mechanism to improve model generalization.

The core concepts and implementation details include:

* **Purpose and Function**: Dropout is used to prevent overfitting by randomly ignoring a subset of hidden layer units during training. This prevents the model from becoming overly reliant on any specific set of units.
* **Placement in Attention**: In transformer architectures like GPT, dropout is typically applied after calculating the attention weights (the most common approach) or after applying those weights to the value vectors.
* **Mathematical Rescaling**: When a dropout rate is applied (e.g., 50%), the remaining active elements are scaled up (e.g., by a factor of 2 calculated by the formula 1 / (1 - `dropout_rate`)) to compensate for the zeroed-out values. This ensures that the average influence of the attention weights remains consistent between the training and inference phases.
* **Training vs. Inference**: It is critical to note that dropout is only active during training and is strictly disabled during the inference (text generation) stage.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.22</strong> Using the causal attention mask (upper left), we apply an additional dropout mask (upper right) to zero out additional attention weights to reduce overfitting during training.
    </figcaption>
</figure>

Ultimately, this minor tweak to the causal attention mechanism ensures the LLM learns more robust features and captures broader semantic patterns instead of memorizing specific training samples.

In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

Note that the resulting dropout outputs may look different depending on your operating system; you can read more about this inconsistency [here on the PyTorch issue tracker](https://github.com/pytorch/pytorch/issues/121595).

### **3.5.3 Implementing a compact causal self-attention class**

This section details the creation of a reusable PyTorch module that integrates causal masking and dropout into the self-attention framework.

The key implementation details and functional improvements include:

* **Batched Input Support**: Unlike previous simplified versions, this `CausalAttention` class is designed to handle three-dimensional tensors (batch size, number of tokens, embedding dimension), making it compatible with the batch outputs generated by standard data loaders.
* **Integrated regularizations**: The class incorporates an `nn.Dropout` layer to prevent overfitting during training and utilizes `nn.Linear` layers for the Query, Key, and Value projections to ensure optimized weight initialization.
* **The Masking Mechanism**: The causal mask is applied to the unnormalized attention scores within the `forward` method, filling future token positions with negative infinity (`-inf`) before the softmax operation to ensure the model only attends to past and current information.
* **Use of `register_buffer`**: The implementation uses `self.register_buffer` to store the attention mask. This ensures the mask is automatically moved to the appropriate device (CPU or GPU) along with the model parameters while remaining a constant that is not updated during training.
* **Resulting Output**: The module transforms input batches into enriched context vectors of the same sequence length, providing the necessary foundation for the multi-head attention components implemented in later sections.

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, trailing `_` ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

Note that dropout is only applied during training, not during inference

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/23.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.23</strong> Here’s what we’ve done so far. We began with a simplified attention mechanism, added trainable weights, and then added a causal attention mask. Next, we will extend the causal attention mechanism and code multi-head attention, which we will use in our LLM.
    </figcaption>
</figure>

## **3.6 Extending single-head attention to multi-head attention**

This section describes the final evolutionary step of the attention module: transition from a single processing unit to multiple independent "heads" operating in parallel.

The key conceptual and technical points include:

* **Defining Multi-Head Attention**: The term "multi-head" refers to dividing the attention mechanism into multiple independent subunits. While a single causal attention module processes input using one set of weights, multi-head attention utilizes several sets of weights simultaneously.
* **Purpose and Benefit**: This parallel structure is crucial for complex pattern recognition. It allows the model to simultaneously attend to information from different "representation subspaces", capturing various aspects and nuances of the input data at once.
* **Implementation Strategy**: The source outlines two ways to build this module:
    1. **Intuitive Approach**: Stacking multiple individual `CausalAttention` modules using a wrapper class.
    2. **Efficient Approach**: A more mathematically integrated version that uses batched matrix multiplications and weight splits to process all heads simultaneously, which is more computationally optimized.

Ultimately, this expansion transforms the single-stream attention process into a robust, parallel framework that serves as a cornerstone for modern transformer-based architectures like GPT.

### **3.6.1 Stacking multiple single-head attention layers**

We simply stack multiple single-head attention modules to obtain a multi-head attention module:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.24</strong> The multi-head attention module includes two single-head attention modules stacked on top of each other. So, instead of using a single matrix for computing the value matrices, in a multi-head attention module with two heads, we now have two value weight matrices. The same applies to the other weight matrices. We obtain two sets of context vectors Z1 and Z2 that we can combine into a single context vector matrix Z.
    </figcaption>
</figure>

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.25</strong> Using the *MultiHeadAttentionWrapper*, we specified the number of attention heads (num_heads). If we set num_heads=2, as in this example, we obtain a tensor with two sets of context vector matrices. In each context vector matrix, the rows represent the context vectors corresponding to the tokens, and the columns correspond to the embedding dimension specified via d_out=4. We concatenate these context vector matrices along the column dimension. Since we have two attention heads and an embedding dimension of 2, the final embedding dimension is 2 × 2 = 4.
    </figcaption>
</figure>

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

#### **Exercise 3.2: Returning two-dimensional embedding vectors**

Change the input arguments for the `MultiHeadAttentionWrapper(..., num_heads=2)` call such that the output context vectors are two-dimensional instead of
four dimensional while keeping the setting `num_heads=2`. Hint: You don’t have to modify the class implementation; you just have to change one of the other input
arguments.

#### **Solution 3.2**

In [ ]:
d_out = 1
mha_sol = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
print(mha_sol(batch))

### **3.6.2 Implementing multi-head attention with weight splits**

This section introduces an integrated `MultiHeadAttention` class that serves as a more computationally efficient alternative to stacking multiple individual attention modules.

The key technical and functional aspects include:

* **Integrated Weight Splits**: Instead of maintaining separate sets of weights for each head, this version initializes one large weight matrix for each of the Query, Key, and Value projections. The input is projected once and then reshaped and transposed (using `.view` and `.transpose`) to internally represent independent heads.
* **Computational Efficiency**: The main advantage is that it requires only one matrix multiplication to compute all keys, queries, or values for the entire multi-head structure. This is significantly faster than the wrapper-based approach, which repeats these expensive operations for every head.
* **Parallelism via Batched Multiplication**: The implementation utilizes batched matrix multiplications to process all attention heads simultaneously across different representation subspaces.
* **Output Projection Layer**: After the context vectors from all heads are combined and flattened, the module passes them through a final output projection layer (`self.out_proj`). While not strictly mandatory, this layer is standard in modern LLMs for effectively merging information captured by parallel heads.

Ultimately, this integrated approach provides the scalable and optimized attention mechanism necessary for high-performance transformer architectures like GPT.

While the above is an intuitive and fully functional implementation of multi-head attention (wrapping the single-head attention `CausalAttention` implementation from earlier), we can write a stand-alone class called `MultiHeadAttention` to achieve the same

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

Note that the above is essentially a rewritten version of `MultiHeadAttentionWrapper` that is more efficient. The resulting output looks a bit different since the random weight initializations differ, but both are fully functional implementations that can be used in the GPT class we will implement in the upcoming chapters

#### **A note about the output dimensions**

* In the `MultiHeadAttention` above, I used `d_out=2` to use the same setting as in the `MultiHeadAttentionWrapper` class earlier
* The `MultiHeadAttentionWrapper`, due the the concatenation, returns the output head dimension `d_out * num_heads` (i.e., `2*2 = 4`)
* However, the `MultiHeadAttention` class (to make it more user-friendly) allows us to control the output head dimension directly via `d_out`; this means, if we set `d_out = 2`, the output head dimension will be 2, regardless of the number of heads
* In hindsight, as readers [pointed out](https://github.com/rasbt/LLMs-from-scratch/pull/859), it may be more intuitive to use `MultiHeadAttention` with `d_out = 4` so that it produces the same output dimensions as `MultiHeadAttentionWrapper` with `d_out = 2`.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 3.26</strong> *MultiHeadAttention* class.
    </figcaption>
</figure>

>**Note:** that in addition, we added a linear projection layer (`self.out_proj `) to the `MultiHeadAttention` class above. This is simply a linear transformation that doesn't change the dimensions. It's a standard convention to use such a projection layer in LLM implementation, but it's not strictly necessary (recent research has shown that it can be removed without affecting the modeling performance; see the further reading section at the end of this chapter)


A compact and efficient implementation of the above class is available as [`torch.nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) class in PyTorch

Since the above implementation may look a bit complex at first glance, let's look at what happens when executing `attn_scores = queries @ keys.transpose(2, 3)`:

In [ ]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

In this case, the matrix multiplication implementation in PyTorch will handle the 4-dimensional input tensor so that the matrix multiplication is carried out between the 2 last dimensions (num_tokens, head_dim) and then repeated for the individual heads

For instance, the following becomes a more compact way to compute the matrix multiplication for each head separately:

In [ ]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

### **Exercise 3.3 Initializing GPT-2 size attention modules**

Using the `MultiHeadAttention` class, initialize a multi-head attention module that has the same number of attention heads as the smallest GPT-2 model (12 attention
heads). Also ensure that you use the respective input and output embedding sizes similar to GPT-2 (768 dimensions). Note that the smallest GPT-2 model supports a context length of 1,024 tokens.

### **Solution 3.3**

In [ ]:
num_heads = 12
d_in, d_out = 768, 768
context_length = 1024

# Using custom class developed here.
mha_gpt2 = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads)
print(mha_gpt2)

# Compute number of trainable weights
print(sum([p.numel() for p in mha_gpt2.parameters() if p.requires_grad]) )

In [ ]:
# Using PyTorch class
mha_gpt2_torch = nn.MultiheadAttention(d_out, num_heads)
print(mha_gpt2_torch)

print(sum([p.numel() for p in mha_gpt2_torch.parameters() if p.requires_grad]))

>**Note:** The GPT-2 model has 117M parameters in total, but as we can see, most of its parameters are not in the multi-head attention module itself.

---

## **Summary and takeaways**

* Attention mechanisms transform input elements into enhanced context vector representations that incorporate information about all inputs.
* A self-attention mechanism computes the context vector representation as a weighted sum over the inputs.
* In a simplified attention mechanism, the attention weights are computed via dot products.
* A dot product is a concise way of multiplying two vectors element-wise and then summing the products.
* Matrix multiplications, while not strictly required, help us implement computations more efficiently and compactly by replacing nested for loops.
* In self-attention mechanisms used in LLMs, also called scaled-dot product attention, we include trainable weight matrices to compute intermediate transformations of the inputs: queries, values, and keys.
* When working with LLMs that read and generate text from left to right, we add a causal attention mask to prevent the LLM from accessing future tokens.
* In addition to causal attention masks to zero-out attention weights, we can add a dropout mask to reduce overfitting in LLMs.
* The attention modules in transformer-based LLMs involve multiple instances of causal attention, which is called multi-head attention.
* We can create a multi-head attention module by stacking multiple instances of causal attention modules.
* A more efficient way of creating multi-head attention modules involves batched matrix multiplications

---

## **Supoplementary Information**

* See the [multihead-attention.ipynb](./multihead-attention.ipynb) code notebook, which is a concise version of the data loader (chapter 2) plus the multi-head attention class that we implemented in this chapter and will need for training the GPT model in upcoming chapters